# 6장. 실행 환경과 의존성 준비

파일 실습에 pandas 설치와 초기화 검사를 추가합니다.

| 연결 기준 | 이번 장 |
| --- | --- |
| 공통 예제 | 동일한 sales.csv의 매출 분석 |
| 새로 배울 기능 | packages·setup_commands·설치 위치와 환경 기록 |
| 이번 실습 조합 | OpenAI-hosted · pandas · 분석 파일과 environment.json |


<!-- colab-link -->
[독립 실행본 다운로드 안내](../../docs/colab.md) · Colab 업로드용


## 학습 안내

| 구분 | 5장 | 이 장 |
| --- | --- | --- |
| 분석 라이브러리 | Python 표준 라이브러리 | 원격 sandbox에 설치한 pandas |
| 환경 준비 | 입력 CSV 배치 | 패키지·입력·초기화 스크립트 준비 |
| 결과 파일 | 분석 JSON·보고서 | 분석 JSON·보고서·환경 정보 |

노트북에서 환경 설정 작성 → `sessions.create()` 요청 → 원격 패키지·파일 준비 → `setup_commands` 실행 → 에이전트 분석 → 결과 다운로드 순서로 진행합니다. 환경 준비 코드를 셀로 나누어도, 로컬 준비 셀은 요청을 구성할 뿐 설치를 시작하지 않습니다.

이번 실행에서는 pandas로 분석합니다. 앞 장의 표준 라이브러리 분석을 자동 재실행하거나 이전 Session의 결과를 가져오지 않습니다. 검증기는 같은 CSV의 기준값과 이번 결과를 대조합니다.

수동 실습은 위에서 아래로 실행합니다. 자동 검증은 `uv run --group dev python scripts/run_intro.py --chapter 6 --live`로 실행합니다. 로컬 패키지 설치, 원격 환경 준비, 모델 응답, 별도 정답 검증을 구분하세요.


**실습 안내:** 각 장은 새 커널에서 독립 실행합니다. 실행 모드·응답과 파일의 차이·검증 기록은 [공통 학습·실습 안내](../../docs/validation.md)를 참고하세요.


## 0. 실습 준비와 실행 모드

로컬에서는 저장소 루트에서 `uv sync --locked`, `uv run jupyter lab`을 실행하고 `.venv` 커널을 선택하세요.
Colab에서는 다운로드 안내의 독립 실행본을 업로드하세요. 준비 셀이 패키지를 직접 설치하고 포함된 코드·데이터를 생성합니다.
실제 API 실습은 **RUN_LIVE=True**로 명시적으로 허용합니다.
키는 `.env.local` 또는 Colab Secrets에서 읽으며 출력하지 않습니다. 라이브 실행에는 비용이 발생합니다.


In [ ]:
import os
import sys

if "google.colab" in sys.modules and os.environ.get("TUTORIAL_OFFLINE_CHECK") != "1":
    raise RuntimeError("This is the local source notebook. Upload the standalone version from the download guide to Colab.")
print("Use the local .venv kernel. Install dependencies with: uv sync --locked")


In [ ]:
import json
from tutorial_helpers.common import load_settings

RUN_LIVE = False
MODEL_OVERRIDE = ""
NUMBER = 6
CHAPTER = "06_environment"
client = None
settings = load_settings(RUN_LIVE, MODEL_OVERRIDE)
print(
    {
        "chapter": NUMBER,
        "live_enabled": settings.enabled,
        "model": settings.model or "Offline mode",
    }
)


## 1. 입력·초기화 코드·환경 요청 준비하기

### 로컬과 원격은 다른 Python 환경입니다

| 위치 | 준비하는 것 |
| --- | --- |
| 로컬 `.venv` 또는 Colab | API 클라이언트·Jupyter·교재 코드 |
| OpenAI-hosted sandbox | 에이전트가 분석에 사용할 pandas |
| 공개 패키지 저장소 | 원격 의존성 설치에 사용하는 패키지 공급원 |

로컬 `uv sync`가 sandbox에 pandas를 설치하는 것은 아닙니다. 원격 버전은 아래 `PANDAS_VERSION` 한 곳에서 정하고 패키지 요청과 초기화 검사가 함께 사용합니다. 원본 CSV는 5장처럼 바이트로 읽습니다. 아직 API 호출이나 원격 설치는 발생하지 않습니다.


In [ ]:
PANDAS_VERSION = "2.3.3"
input_path = settings.root / "data/sales.csv"
csv_bytes = input_path.read_bytes()
csv_text = csv_bytes.decode("utf-8")
print(csv_text)
print("Requested sandbox pandas version:", PANDAS_VERSION)


### 초기화 스크립트 정의하기

다음 문자열은 sandbox에 공급할 `check_setup.py`의 내용입니다. 로컬에서 정의·표시할 뿐 pandas를 import하거나 실행하지 않습니다.

1. sandbox의 입력 CSV 존재와 pandas 버전을 확인합니다.
2. 입력 열 구조를 확인합니다.
3. pandas·Python 버전과 입력 행 수를 `/workspace/setup_info.json`에 기록합니다.

초기화 코드는 매출 정답을 계산하지 않습니다. 조건이 맞지 않으면 예외로 종료하며, 초기화 실패를 빈 보고서로 바꾸어 성공 처리하지 않습니다.


In [ ]:
from textwrap import dedent

setup_source = dedent(f"""
    from pathlib import Path
    import json
    import platform
    import pandas as pd

    csv_path = Path('/workspace/sales.csv')
    if not csv_path.is_file():
        raise FileNotFoundError('CSV was not prepared before setup')
    if pd.__version__ != {PANDAS_VERSION!r}:
        raise RuntimeError('Unexpected pandas version')
    frame = pd.read_csv(csv_path)
    if list(frame.columns) != ['date', 'product', 'quantity', 'unit_price']:
        raise ValueError('Unexpected CSV columns')

    info = dict(
        pandas_version=pd.__version__,
        python_version=platform.python_version(),
        input_rows=len(frame),
    )
    Path('/workspace/setup_info.json').write_text(json.dumps(info), encoding='utf-8')
""").lstrip()
print(setup_source)


### API에 보낼 환경 설정 구성하기

| 필드 | 이 실습의 역할 |
| --- | --- |
| `files` | CSV와 초기화 스크립트 배치 |
| `packages.python` | 원격 pandas 설치 요청 |
| `network` | sandbox의 접근 도메인 제한 |
| `setup_commands` | 출력 디렉터리 생성 후 초기화 검사 실행 |

파일과 패키지가 준비된 뒤 setup 명령이 지정 순서로 실행됩니다. 아래 딕셔너리는 이 요청의 최종 설정이며, 다음 실행 단계에서 API로 전달합니다. 공개 패키지 저장소 도메인 허용은 이번 예제의 설정입니다. 다른 패키지·사내 레지스트리의 네트워크·인증까지 보장하는 설정은 아닙니다. 사내 실행 환경은 [12장](12_self_hosted.ipynb)에서 다룹니다.

API 키를 환경 변수나 setup 명령의 평문에 추가하지 마세요.


In [ ]:
import base64

environment = {
    "type": "openai_hosted",
    "container_size": "small",
    "files": [
        {"type": "inline", "path": "/workspace/sales.csv",
         "data": base64.b64encode(csv_bytes).decode("ascii")},
        {"type": "inline", "path": "/workspace/check_setup.py",
         "data": base64.b64encode(setup_source.encode("utf-8")).decode("ascii")},
    ],
    "packages": {"python": [f"pandas=={PANDAS_VERSION}"]},
    "network": {
        "access": "restricted",
        "allowed_domains": ["pypi.org", "files.pythonhosted.org"],
    },
    "setup_commands": [
        {"command": "mkdir -p /workspace/outputs", "cwd": "/workspace"},
        {"command": "python /workspace/check_setup.py", "cwd": "/workspace"},
    ],
}
print({
    "input_paths": [file["path"] for file in environment["files"]],
    "packages": environment["packages"],
    "setup_commands": environment["setup_commands"],
    "network": environment["network"],
})


### pandas 분석과 결과 게시 요청 구성하기

에이전트에게 입력 파일을 pandas로 읽어 분석 JSON·한국어 보고서를 작성하고, 초기화 기록을 그대로 `/workspace/outputs/environment.json`에 복사하도록 요청합니다. `METRICS_CONTRACT`는 최종 JSON 형식이며 실제 입력 메시지에 조합합니다.

환경 정보는 초기화 기록을 에이전트가 복사한 자료입니다. 플랫폼이 별도로 보증한 변조 불가능한 실행 증명으로 해석하지 않습니다.


In [ ]:
BASE_INSTRUCTIONS = (
    "You are a careful sales analyst. Use Python "
    "for all calculations. Use only the provided "
    "CSV; never invent rows. Do not browse the web"
    " or delegate. Keep progress commentary short."
    " Return only the JSON object requested by the"
    " user as the final answer, with no Markdown "
    "fences."
)
METRICS_CONTRACT = (
    'Return exactly {"row_count": integer, '
    '"total_quantity": integer, "total_revenue": '
    'integer, "by_product": {product_name: '
    "integer_revenue}}. Revenue is quantity "
    "multiplied by unit_price. Do not round "
    "integer amounts."
)
task = (
    "Use pandas to read /workspace/sales.csv and compute the summary. "
    "Write /workspace/outputs/summary.json containing exactly the requested summary. "
    "Also write /workspace/outputs/report.md in Korean, starting with '# 매출 분석 보고서'. "
    "Include row count, total quantity, total revenue as digits, and every product's revenue. "
    "Read both files back and verify they exist. "
    "Copy /workspace/setup_info.json verbatim to /workspace/outputs/environment.json. "
    + METRICS_CONTRACT
)


## 2. 환경 준비를 포함한 에이전트 실행

환경 설정을 `sessions.create()`로 전달하면 원격 준비와 초기화를 거쳐 분석이 진행됩니다. 패키지 준비로 첫 텍스트 응답까지 시간이 걸릴 수 있습니다.

| 관찰할 이벤트 | 읽는 방법 |
| --- | --- |
| `agent.session.created` | 생성된 Session ID·환경 ID 기록 |
| `agent.session.environment.pending` | 환경 준비 관련 진행 정보 |
| `agent.session.environment.ready` | 환경의 연결 준비 상태. 버전·계산 정확성의 증거와 구분 |
| `agent.session.environment.failed` | 환경 실패. 성공으로 처리하지 않음 |

수신한 준비 관련 이벤트의 종류와 환경 ID만 표시합니다. 이벤트 수·순서를 고정하거나 특정 준비 이벤트를 반드시 받아야 한다고 가정하지 않습니다. 전체 이벤트나 Trace를 추가 수집하지 않습니다.

실행부터 결과 확인까지 각 단계는 `cleanup_on_error()`로 오류·일반적인 인터럽트 시 정리를 시도합니다. 정상 경로에서는 **6절 정리 셀까지 실행**하고, 중간에 학습을 멈출 때도 정리 셀을 실행하세요. 강제 커널 종료는 자동 정리를 보장하지 못합니다.


#### API 실행 허용 확인

설정 셀에서 `RUN_LIVE=True`를 선택한 뒤 이 준비 셀을 실행하세요. 기본 False에서는 여기서 오류 메시지와 함께 멈춥니다. 이후 API 셀에는 실행 여부 분기를 반복하지 않습니다. `requires_api` 태그는 오프라인 검증기용이며 Jupyter 자체의 접근 제어가 아닙니다.


In [ ]:
from openai import OpenAI

client = None
if not settings.enabled:
    raise RuntimeError(
        "Set RUN_LIVE=True to run the API example. Use the validation script for offline checks."
    )
client = OpenAI(api_key=settings.api_key, timeout=90.0, max_retries=0)


### 이번 실행 상태 준비

`output_dir`는 복구 기록과 다운로드를 저장할 로컬 경로입니다. 새 실행을 시작할 때 상태 준비 셀을 한 번만 실행하세요. 진행 중 다시 실행하면 자원 식별자를 잃을 수 있습니다.

타이머는 API 실행 직전에 시작하여 셀 사이 학습 시간을 제외합니다. 600초 제한은 이벤트를 받을 때 확인하므로 무응답 중 강제 중단하는 전체 타이머가 아닙니다. 클라이언트 HTTP timeout은 90초입니다.


In [ ]:
from tutorial_helpers.common import (
    new_session_metadata, cleanup_resources, cleanup_on_error, check_event, safe_error,
)
from uuid import uuid4
import time


In [ ]:
run_id = uuid4().hex
output_dir = settings.root / "outputs/intro" / CHAPTER / run_id
output_dir.mkdir(parents=True, exist_ok=True)
sessions, agents, attempts = {}, {}, []
cleanup_errors = []
result = turn = None
downloads = output_dir / "downloads"
downloads_complete = False


### 세션을 만들고 준비·분석 진행 관찰하기

이 셀은 새 작업을 제출하므로 중복 실행을 차단합니다. 텍스트 delta를 표시하면서 `with_result_collection()`으로 최종 응답을 모읍니다. 실패·취소는 `check_event()`로 처리합니다.

`get_final_result()` 결과와 저장된 root Turn의 완료 상태를 확인하고 최종 분석 JSON을 표시합니다. 준비 이벤트, Turn 완료, JSON 파싱 성공만으로 계산값까지 검증됐다고 판단하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if attempts:
        raise RuntimeError("This run already attempted session creation. Do not resubmit.")
    started = time.monotonic()
    with client.beta.agents.sessions.create(
        agent={"model": settings.model, "instructions": BASE_INSTRUCTIONS},
        environment=environment,
        input=task,
        metadata=new_session_metadata(CHAPTER, run_id, attempts=attempts, output_dir=output_dir),
        stream=True,
    ).with_result_collection() as stream:
        for event in stream:
            if event.type == "agent.session.created":
                sessions[event.session.id] = {
                    "deleted": False,
                    "environment_id": getattr(event.session.environment, "id", None),
                }
                print("Session:", event.session.id,
                      "Environment:", sessions[event.session.id]["environment_id"])
            elif event.type in {
                "agent.session.environment.pending", "agent.session.environment.ready",
                "agent.session.environment.failed",
            }:
                print("Environment event:", event.type, "ID:", event.environment.id, flush=True)
                check_event(event)
            elif event.type == "agent.session.turn.output_text.delta":
                print(event.delta, end="", flush=True)
            elif event.type == "agent.session.requires_action":
                raise RuntimeError("Unexpected external action in this hosted exercise")
            else:
                check_event(event)
            if time.monotonic() - started > 600:
                raise TimeoutError("Event-observed execution deadline exceeded")
        result = stream.get_final_result()
        sessions.setdefault(result.session_id, {"deleted": False})

    turn = client.beta.agents.sessions.turns.retrieve(
        result.turn_id, session_id=result.session_id, timeout=15
    )
    if result.turn.status != "completed" or turn.status != "completed":
        raise RuntimeError("Root turn completion was not confirmed")
    print("\nCompleted analysis:")
    print({"session_id": result.session_id, "turn_id": result.turn_id, "status": turn.status})
    print(json.dumps(json.loads(result.output_text), indent=2))


## 3. 이번 Turn의 게시 파일 확인하기

05장처럼 `artifacts.list()`를 SDK 페이지 반복으로 읽고 결과의 Session·Turn ID에 맞는 파일만 표시합니다. 파일 ID·경로·크기는 게시 정보이며 실제 다운로드나 내용 정확성과 구분합니다.

| 결과 파일 | 내용 |
| --- | --- |
| `/workspace/outputs/summary.json` | 분석 집계 |
| `/workspace/outputs/report.md` | 한국어 매출 보고서 |
| `/workspace/outputs/environment.json` | 초기화 기록의 사본 |

sandbox의 작업 파일·게시된 artifact·로컬 다운로드는 서로 다른 대상입니다. 필요한 파일은 세션 삭제 전에 내려받습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before inspecting published files.")
    published = list(client.beta.agents.sessions.artifacts.list(result.session_id, timeout=15))
    current_artifacts = [a for a in published
                         if a.session_id == result.session_id and a.turn_id == result.turn_id]
    print("Files published by this turn:")
    print(json.dumps([
        {"id": a.id, "path": a.path, "turn_id": a.turn_id, "bytes": a.size_bytes}
        for a in current_artifacts
    ], indent=2))


## 4. 세 결과 파일 다운로드하기

`artifacts.for_result(result)`로 완료한 Session·Turn에 범위를 고정하고 정확한 원격 경로를 지정합니다. SDK가 누락·중복 경로를 오류로 처리하므로 별도 선택 helper를 만들지 않습니다. 같은 다운로드 동작을 세 파일에 반복하되 API 호출은 셀에 직접 표시합니다.

`to=`는 로컬 경로입니다. 다운로드가 반환한 artifact ID·Turn·원격 경로와 실제 로컬 저장 경로·크기를 확인하세요. 세 파일을 모두 받은 뒤에만 완료 상태로 표시합니다.

재실행은 세션 정리 전에 한정하세요. 부분 전송이나 마지막 파일 실패도 성공으로 처리하지 않습니다. 오류 시 정리를 시도하므로 같은 세션에서 재시도할 수 있다고 보장하지 않으며, 분석을 자동으로 재실행하지 않습니다.


In [ ]:
with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if turn is None or turn.status != "completed":
        raise RuntimeError("Complete the analysis before downloading artifacts.")
    downloads_complete = False
    downloads.mkdir(exist_ok=True)
    scoped = client.beta.agents.sessions.artifacts.for_result(result)
    for name in ("summary.json", "report.md", "environment.json"):
        local_path = downloads / name
        artifact = scoped.download(f"/workspace/outputs/{name}", to=local_path, timeout=30)
        print({"artifact_id": artifact.id, "turn_id": artifact.turn_id,
               "sandbox_path": artifact.path, "local_path": str(local_path),
               "local_bytes": local_path.stat().st_size})
    downloads_complete = True


## 5. 환경 정보와 실제 분석 결과 읽기

| 기록 | 생성 방식 |
| --- | --- |
| `/workspace/setup_info.json` | setup 스크립트가 버전·입력 준비 상태를 기록 |
| `/workspace/outputs/environment.json` | 에이전트가 setup 기록을 그대로 복사하도록 요청 |
| 로컬 `environment.json` | 해당 Turn의 artifact를 다운로드한 파일 |

아래 셀은 요청한 pandas 버전과 기록된 버전을 비교하고 Python 버전·입력 행 수를 표시합니다. 버전 불일치는 오류로 처리합니다. 환경 파일은 복사된 기록이므로 별도 플랫폼 검증이나 변조 방지를 보장하지 않습니다.

로컬 분석 JSON과 보고서도 읽습니다. 이 정상 경로는 새 API 요청 없이 로컬 파일을 사용합니다. 버전 비교는 이 장의 환경 학습 기능이고, CSV 정답·보고서 내용·입력 행 수 대조는 별도 검증기가 수행합니다. 자동 검증 통과와 수동 출력 확인을 구분하세요.


In [ ]:
from IPython.display import Markdown, display

with cleanup_on_error(client, sessions, agents, attempts, errors=cleanup_errors):
    if not downloads_complete:
        raise RuntimeError("Complete all three downloads before reading local results.")
    environment_info = json.loads((downloads / "environment.json").read_text(encoding="utf-8"))
    pandas_version_matches = environment_info["pandas_version"] == PANDAS_VERSION
    print({"requested_pandas_version": PANDAS_VERSION,
           "reported_pandas_version": environment_info["pandas_version"],
           "pandas_version_matches": pandas_version_matches,
           "python_version": environment_info["python_version"],
           "input_rows": environment_info["input_rows"]})
    if not pandas_version_matches:
        raise RuntimeError("Reported pandas version differs from the requested version")
    downloaded_summary = json.loads((downloads / "summary.json").read_text(encoding="utf-8"))
    report_text = (downloads / "report.md").read_text(encoding="utf-8")
    print("Downloaded summary.json:")
    print(json.dumps(downloaded_summary, indent=2))
    display(Markdown(report_text))


## 6. 세션 정리와 로컬 결과 보관

중간에 학습을 멈추더라도 정리 셀을 실행하세요. 스트림 종료는 세션 삭제가 아닙니다. 정리 helper는 이번 실행의 metadata로 소유 세션을 찾아 필요하면 취소한 뒤 삭제·재조회 `404`로 API상 부재를 확인합니다.

출력의 `deleted`·`verified_absent`를 확인합니다. 물리적 sandbox 정리가 동기적으로 완료됐다는 뜻은 아닙니다. 원래 실행·검증 오류가 있으면 정리 오류가 그 원인을 가리지 않도록 처리합니다.

검증 도구는 삭제 전에 이력·파일 내용 검사를 추가하고 실패해도 정리를 시도합니다. 세션 삭제 후에도 로컬 세 파일과 결과 객체는 남습니다. 삭제 후 서버에서 다시 받을 수 있다고 가정하지 마세요.


In [ ]:
try:
    print("Cleaning up this run's session...", flush=True)
finally:
    original_error = sys.exc_info()[1]
    try:
        cleanup_errors[:] = cleanup_resources(client, sessions, agents, attempts)
    except Exception as cleanup_error:
        cleanup_errors.append(safe_error(cleanup_error))
    if cleanup_errors:
        if original_error is None:
            raise RuntimeError("Resource cleanup incomplete; inspect local records")
        original_error.add_note("Resource cleanup incomplete; inspect local records")
if not all(owned.get("verified_absent") for owned in sessions.values()):
    raise RuntimeError("Session absence was not confirmed")
print({"session_cleanup": sessions})


## 7. 재현성·네트워크·실패 시 조치

`uv.lock`은 로컬 의존성을 고정하고, 원격 pandas 버전은 `environment.packages.python`에서 별도로 지정합니다. 이 버전 고정만으로 전이 의존성·운영체제 이미지·외부 API·모델 응답까지 동일해지는 것은 아닙니다.

| 실패 단계 | 확인할 내용 |
| --- | --- |
| 패키지 준비 | 지정 버전·패키지 저장소 접근·네트워크 설정 |
| setup 검사 | CSV 경로·열 구조·pandas 버전 조건 |
| 분석 Turn | 준비와 구분하여 실패·취소 상태 확인 |
| 다운로드 | 누락 artifact·정확한 Turn/경로·부분 파일과 정리 결과 |
| 강제 커널 종료 | Session ID와 `outputs/intro/<chapter>/<run_id>/creation.json`으로 이번 실행 식별 |

키나 원본 오류 body를 출력하지 마세요. 같은 실패 요청을 무제한 반복하거나 다른 환경 종류로 자동 전환하지 않습니다. 401/403은 인증·프로젝트 권한, 429는 API 제한을 확인합니다. 검증 도구로 실행했다면 로컬 `summary.json`도 참고하세요. 복구 절차는 [빠른 시작](../quickstart.ipynb)을 참고하세요.


## 입문 과정 완료 기준과 다음 장

- 로컬 패키지와 원격 패키지의 설치 위치를 구분하고, 패키지·파일 준비 후 setup 검사를 거쳐 분석하는 흐름을 설명할 수 있습니다.
- 최종 응답·게시 정보·세 로컬 파일·요청/기록 버전 비교·세션 삭제 결과를 구분해 읽을 수 있습니다.
- 수동 실습은 파일과 출력 확인이며, 표준 라이브러리와 pandas의 계산 정확성 대조는 동일 CSV 기준값을 사용하는 별도 검증기의 역할입니다. 앞 장을 다시 실행할 필요는 없습니다.

[7장 Function Tool](07_function_tools.ipynb)에서는 sandbox 패키지 안에서 계산하는 흐름에 애플리케이션의 업무 함수를 연결합니다. 함수 실행 위치와 sandbox 실행 위치의 차이를 이어서 배웁니다.


## 공식 참고 자료

### Agents API 전용 문서

- [OpenAI-hosted sandboxes](https://developers.openai.com/api/docs/guides/agents-api/environments/openai-hosted) — packages·setup_commands·초기 파일·네트워크 설정.
- [Sandbox security](https://developers.openai.com/api/docs/guides/agents-api/environments/security) — 격리·네트워크·실행 권한 검토·인증 정보 분리.

링크 확인: 2026-11-03. 문서 링크 확인은 API 실습 실행 검증과 별개입니다.
